In [2]:
import io
import math
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Tuple, Union

import fitz  # PyMuPDF
from PIL import Image, ImageTk, ImageGrab, ImageOps, ImageDraw, ImageFont

try:
    from tkinterdnd2 import DND_FILES, TkinterDnD
    DND_AVAILABLE = True
except Exception:
    DND_AVAILABLE = False

import tkinter as tk
from tkinter import ttk, filedialog, messagebox, simpledialog


APP_TITLE = "FigureBoard"
DEFAULT_CANVAS_W = 1600
DEFAULT_CANVAS_H = 1000
DEFAULT_BG = "white"
DEFAULT_OUTER_MARGIN = 30
DEFAULT_GAP_X = 25
DEFAULT_GAP_Y = 25
DEFAULT_LABEL_FONT_SIZE = 28
DEFAULT_TEXT_FONT_SIZE = 22
HANDLE_SIZE = 10
MIN_PANEL_SIZE = 40


def parse_dnd_files(data: str) -> List[str]:
    files = []
    current = ""
    brace = False
    for ch in data:
        if ch == "{":
            brace = True
            current = ""
        elif ch == "}":
            brace = False
            if current:
                files.append(current)
                current = ""
        elif ch == " " and not brace:
            if current:
                files.append(current)
                current = ""
        else:
            current += ch
    if current:
        files.append(current)
    return files


def fit_size_keep_aspect(src_w: int, src_h: int, max_w: int, max_h: int) -> Tuple[int, int]:
    if src_w <= 0 or src_h <= 0:
        return max(1, max_w), max(1, max_h)
    scale = min(max_w / src_w, max_h / src_h)
    return max(1, int(src_w * scale)), max(1, int(src_h * scale))


def next_panel_label(index: int) -> str:
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    result = ""
    i = index
    while True:
        result = alphabet[i % 26] + result
        i = i // 26 - 1
        if i < 0:
            break
    return result


def get_default_font(size: int):
    try:
        return ImageFont.truetype("arial.ttf", size=size)
    except Exception:
        try:
            return ImageFont.truetype("DejaVuSans-Bold.ttf", size=size)
        except Exception:
            return ImageFont.load_default()


@dataclass
class PanelItem:
    id: int
    kind: str = "panel"
    source_path: Optional[str] = None
    pil_image: Optional[Image.Image] = None
    original_size: Tuple[int, int] = (100, 100)
    x: int = 50
    y: int = 50
    w: int = 200
    h: int = 150
    label: str = ""
    show_label: bool = True
    label_font_size: int = DEFAULT_LABEL_FONT_SIZE
    label_offset_x: int = 10
    label_offset_y: int = 10
    border_width: int = 0
    border_color: str = "black"
    z_index: int = 0
    tk_preview: Optional[ImageTk.PhotoImage] = None

    def bbox(self):
        return self.x, self.y, self.x + self.w, self.y + self.h

    def contains(self, px: int, py: int) -> bool:
        return self.x <= px <= self.x + self.w and self.y <= py <= self.y + self.h

    def resize_handle_hit(self, px: int, py: int) -> bool:
        return (self.x + self.w - HANDLE_SIZE <= px <= self.x + self.w + HANDLE_SIZE and
                self.y + self.h - HANDLE_SIZE <= py <= self.y + self.h + HANDLE_SIZE)


@dataclass
class TextItem:
    id: int
    kind: str = "text"
    text: str = "Text"
    x: int = 60
    y: int = 60
    w: int = 250
    h: int = 80
    font_size: int = DEFAULT_TEXT_FONT_SIZE
    bold: bool = False
    fill: str = "black"
    align: str = "left"
    background: Optional[str] = None
    outline: Optional[str] = None
    z_index: int = 0

    def bbox(self):
        return self.x, self.y, self.x + self.w, self.y + self.h

    def contains(self, px: int, py: int) -> bool:
        return self.x <= px <= self.x + self.w and self.y <= py <= self.y + self.h

    def resize_handle_hit(self, px: int, py: int) -> bool:
        return (self.x + self.w - HANDLE_SIZE <= px <= self.x + self.w + HANDLE_SIZE and
                self.y + self.h - HANDLE_SIZE <= py <= self.y + self.h + HANDLE_SIZE)


CanvasItem = Union[PanelItem, TextItem]


class FigureBoardApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title(APP_TITLE)
        self.root.geometry("1700x1050")

        self.canvas_w = DEFAULT_CANVAS_W
        self.canvas_h = DEFAULT_CANVAS_H
        self.bg_color = DEFAULT_BG

        self.items: List[CanvasItem] = []
        self.next_id = 1
        self.selected_id: Optional[int] = None
        self.drag_mode: Optional[str] = None
        self.drag_start = (0, 0)
        self.orig_geom = None

        self.outer_margin = tk.IntVar(value=DEFAULT_OUTER_MARGIN)
        self.gap_x = tk.IntVar(value=DEFAULT_GAP_X)
        self.gap_y = tk.IntVar(value=DEFAULT_GAP_Y)
        self.default_label_size = tk.IntVar(value=DEFAULT_LABEL_FONT_SIZE)
        self.default_text_size = tk.IntVar(value=DEFAULT_TEXT_FONT_SIZE)
        self.auto_trim_on_export = tk.BooleanVar(value=True)
        self.keep_label_order = tk.BooleanVar(value=True)
        self.show_grid = tk.BooleanVar(value=False)

        self.status_var = tk.StringVar(value="Ready")

        self._build_ui()
        self._bind_events()
        self.redraw()

    def _build_ui(self):
        outer = ttk.Frame(self.root)
        outer.pack(fill="both", expand=True)

        left = ttk.Frame(outer, width=260)
        left.pack(side="left", fill="y", padx=6, pady=6)

        center = ttk.Frame(outer)
        center.pack(side="left", fill="both", expand=True, padx=6, pady=6)

        right = ttk.Frame(outer, width=280)
        right.pack(side="right", fill="y", padx=6, pady=6)

        # Left controls
        import_box = ttk.LabelFrame(left, text="Import")
        import_box.pack(fill="x", pady=(0, 8))
        ttk.Button(import_box, text="Add image/PDF files", command=self.add_files).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Paste from clipboard", command=self.paste_from_clipboard).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Add text box", command=self.add_text_box).pack(fill="x", padx=6, pady=4)
        ttk.Button(import_box, text="Add caption box", command=lambda: self.add_text_box(default_text="Caption")).pack(fill="x", padx=6, pady=4)

        arrange_box = ttk.LabelFrame(left, text="Arrange")
        arrange_box.pack(fill="x", pady=(0, 8))
        ttk.Button(arrange_box, text="Auto grid", command=self.auto_grid).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Distribute horizontally", command=self.distribute_h).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Distribute vertically", command=self.distribute_v).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Align left", command=self.align_left).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Align top", command=self.align_top).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Same widths", command=self.same_widths).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Same heights", command=self.same_heights).pack(fill="x", padx=6, pady=4)
        ttk.Button(arrange_box, text="Tight crop to content", command=self.crop_canvas_to_content).pack(fill="x", padx=6, pady=4)

        label_box = ttk.LabelFrame(left, text="Labels")
        label_box.pack(fill="x", pady=(0, 8))
        ttk.Button(label_box, text="Regenerate A, B, C...", command=self.regenerate_labels).pack(fill="x", padx=6, pady=4)
        ttk.Button(label_box, text="Toggle labels on/off", command=self.toggle_labels).pack(fill="x", padx=6, pady=4)
        ttk.Label(label_box, text="Default label size").pack(anchor="w", padx=6, pady=(6, 2))
        ttk.Spinbox(label_box, from_=8, to=120, textvariable=self.default_label_size, width=8,
                    command=self.apply_default_label_size).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Checkbutton(label_box, text="Keep labels by add order", variable=self.keep_label_order).pack(anchor="w", padx=6, pady=4)

        canvas_box = ttk.LabelFrame(left, text="Canvas")
        canvas_box.pack(fill="x", pady=(0, 8))
        row = ttk.Frame(canvas_box)
        row.pack(fill="x", padx=6, pady=4)
        ttk.Label(row, text="W").pack(side="left")
        self.canvas_w_entry = ttk.Entry(row, width=8)
        self.canvas_w_entry.insert(0, str(self.canvas_w))
        self.canvas_w_entry.pack(side="left", padx=(4, 10))
        ttk.Label(row, text="H").pack(side="left")
        self.canvas_h_entry = ttk.Entry(row, width=8)
        self.canvas_h_entry.insert(0, str(self.canvas_h))
        self.canvas_h_entry.pack(side="left", padx=4)
        ttk.Button(canvas_box, text="Apply canvas size", command=self.apply_canvas_size).pack(fill="x", padx=6, pady=4)

        ttk.Label(canvas_box, text="Outer margin").pack(anchor="w", padx=6, pady=(6, 2))
        ttk.Spinbox(canvas_box, from_=0, to=500, textvariable=self.outer_margin, width=8).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Label(canvas_box, text="Gap X").pack(anchor="w", padx=6, pady=(2, 2))
        ttk.Spinbox(canvas_box, from_=0, to=500, textvariable=self.gap_x, width=8).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Label(canvas_box, text="Gap Y").pack(anchor="w", padx=6, pady=(2, 2))
        ttk.Spinbox(canvas_box, from_=0, to=500, textvariable=self.gap_y, width=8).pack(anchor="w", padx=6, pady=(0, 6))
        ttk.Checkbutton(canvas_box, text="Show grid", variable=self.show_grid, command=self.redraw).pack(anchor="w", padx=6, pady=4)

        export_box = ttk.LabelFrame(left, text="Export")
        export_box.pack(fill="x", pady=(0, 8))
        ttk.Checkbutton(export_box, text="Trim whitespace on export", variable=self.auto_trim_on_export).pack(anchor="w", padx=6, pady=4)
        ttk.Button(export_box, text="Export PNG", command=lambda: self.export_image("PNG")).pack(fill="x", padx=6, pady=4)
        ttk.Button(export_box, text="Export TIFF", command=lambda: self.export_image("TIFF")).pack(fill="x", padx=6, pady=4)
        ttk.Button(export_box, text="Export PDF", command=self.export_pdf).pack(fill="x", padx=6, pady=4)

        # Center: canvas with scrollbars
        canvas_frame = ttk.Frame(center)
        canvas_frame.pack(fill="both", expand=True)

        self.hbar = ttk.Scrollbar(canvas_frame, orient="horizontal")
        self.vbar = ttk.Scrollbar(canvas_frame, orient="vertical")
        self.canvas = tk.Canvas(canvas_frame, bg=self.bg_color,
                                xscrollcommand=self.hbar.set,
                                yscrollcommand=self.vbar.set,
                                highlightthickness=1,
                                highlightbackground="#888")
        self.hbar.config(command=self.canvas.xview)
        self.vbar.config(command=self.canvas.yview)

        self.hbar.pack(side="bottom", fill="x")
        self.vbar.pack(side="right", fill="y")
        self.canvas.pack(side="left", fill="both", expand=True)
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))

        if DND_AVAILABLE:
            try:
                self.canvas.drop_target_register(DND_FILES)
                self.canvas.dnd_bind("<<Drop>>", self.on_drop)
            except Exception:
                pass

        # Right: properties
        prop_box = ttk.LabelFrame(right, text="Selected item")
        prop_box.pack(fill="x", pady=(0, 8))

        self.sel_type_var = tk.StringVar(value="-")
        self.sel_label_var = tk.StringVar(value="")
        self.sel_text_var = tk.StringVar(value="")
        self.sel_x_var = tk.StringVar(value="0")
        self.sel_y_var = tk.StringVar(value="0")
        self.sel_w_var = tk.StringVar(value="0")
        self.sel_h_var = tk.StringVar(value="0")
        self.sel_font_var = tk.StringVar(value=str(DEFAULT_TEXT_FONT_SIZE))
        self.sel_show_label_var = tk.BooleanVar(value=True)

        ttk.Label(prop_box, text="Type").grid(row=0, column=0, sticky="w", padx=6, pady=4)
        ttk.Label(prop_box, textvariable=self.sel_type_var).grid(row=0, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Label").grid(row=1, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_label_var).grid(row=1, column=1, sticky="ew", padx=6, pady=4)

        ttk.Label(prop_box, text="Text").grid(row=2, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_text_var).grid(row=2, column=1, sticky="ew", padx=6, pady=4)

        ttk.Label(prop_box, text="X").grid(row=3, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_x_var, width=10).grid(row=3, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Y").grid(row=4, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_y_var, width=10).grid(row=4, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="W").grid(row=5, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_w_var, width=10).grid(row=5, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="H").grid(row=6, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_h_var, width=10).grid(row=6, column=1, sticky="w", padx=6, pady=4)

        ttk.Label(prop_box, text="Font size").grid(row=7, column=0, sticky="w", padx=6, pady=4)
        ttk.Entry(prop_box, textvariable=self.sel_font_var, width=10).grid(row=7, column=1, sticky="w", padx=6, pady=4)

        ttk.Checkbutton(prop_box, text="Show panel label", variable=self.sel_show_label_var).grid(
            row=8, column=0, columnspan=2, sticky="w", padx=6, pady=4
        )

        ttk.Button(prop_box, text="Apply changes", command=self.apply_selected_properties).grid(
            row=9, column=0, columnspan=2, sticky="ew", padx=6, pady=6
        )
        ttk.Button(prop_box, text="Bring to front", command=self.bring_to_front).grid(
            row=10, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )
        ttk.Button(prop_box, text="Send to back", command=self.send_to_back).grid(
            row=11, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )
        ttk.Button(prop_box, text="Duplicate", command=self.duplicate_selected).grid(
            row=12, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )
        ttk.Button(prop_box, text="Delete", command=self.delete_selected).grid(
            row=13, column=0, columnspan=2, sticky="ew", padx=6, pady=4
        )

        prop_box.columnconfigure(1, weight=1)

        help_box = ttk.LabelFrame(right, text="Tips")
        help_box.pack(fill="x", pady=(0, 8))
        help_text = (
            "Mouse:\n"
            "- Click item to select\n"
            "- Drag to move\n"
            "- Drag bottom-right handle to resize\n\n"
            "Keyboard:\n"
            "- Delete: remove selected\n"
            "- Ctrl+V: paste image\n"
            "- Ctrl+O: open files\n"
            "- Ctrl+E: export PNG\n"
            "- Ctrl+G: auto grid\n"
        )
        ttk.Label(help_box, text=help_text, justify="left").pack(anchor="w", padx=8, pady=8)

        status = ttk.Label(self.root, textvariable=self.status_var, anchor="w")
        status.pack(fill="x", side="bottom")

    def _bind_events(self):
        self.canvas.bind("<Button-1>", self.on_canvas_press)
        self.canvas.bind("<B1-Motion>", self.on_canvas_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_canvas_release)
        self.canvas.bind("<Double-Button-1>", self.on_double_click)

        self.root.bind("<Delete>", lambda e: self.delete_selected())
        self.root.bind("<Control-v>", lambda e: self.paste_from_clipboard())
        self.root.bind("<Control-o>", lambda e: self.add_files())
        self.root.bind("<Control-e>", lambda e: self.export_image("PNG"))
        self.root.bind("<Control-g>", lambda e: self.auto_grid())

    def set_status(self, text: str):
        self.status_var.set(text)

    def apply_canvas_size(self):
        try:
            self.canvas_w = max(100, int(self.canvas_w_entry.get()))
            self.canvas_h = max(100, int(self.canvas_h_entry.get()))
        except ValueError:
            messagebox.showerror("Invalid size", "Canvas width/height must be integers.")
            return
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.redraw()
        self.set_status(f"Canvas set to {self.canvas_w} x {self.canvas_h}")

    def get_next_id(self) -> int:
        out = self.next_id
        self.next_id += 1
        return out

    def add_files(self):
        filetypes = [
            ("Supported", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp *.gif *.pdf"),
            ("Images", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp *.gif"),
            ("PDF files", "*.pdf"),
            ("All files", "*.*"),
        ]
        paths = filedialog.askopenfilenames(title="Select images or PDFs", filetypes=filetypes)
        if not paths:
            return
        self.load_paths(list(paths))
        self.redraw()

    def on_drop(self, event):
        paths = parse_dnd_files(event.data)
        if paths:
            self.load_paths(paths)
            self.redraw()

    def load_paths(self, paths: List[str]):
        added = 0
        for path in paths:
            path = str(path).strip()
            if not path:
                continue
            ext = Path(path).suffix.lower()
            try:
                if ext == ".pdf":
                    added += self._load_pdf(path)
                else:
                    self._load_image_path(path)
                    added += 1
            except Exception as e:
                messagebox.showerror("Import error", f"Could not open:\n{path}\n\n{e}")
        if added:
            self.regenerate_labels()
            self.set_status(f"Added {added} item(s)")
            self.auto_grid_if_reasonable()

    def _load_image_path(self, path: str):
        img = Image.open(path).convert("RGBA")
        self._add_panel_from_pil(img, source_path=path)

    def _load_pdf(self, path: str) -> int:
        doc = fitz.open(path)
        count = 0
        for pno in range(len(doc)):
            page = doc[pno]
            pix = page.get_pixmap(matrix=fitz.Matrix(2.5, 2.5), alpha=False)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGBA")
            self._add_panel_from_pil(img, source_path=f"{path} [page {pno+1}]")
            count += 1
        doc.close()
        return count

    def paste_from_clipboard(self):
        try:
            data = ImageGrab.grabclipboard()
        except Exception as e:
            messagebox.showerror("Clipboard error", f"Could not access clipboard.\n\n{e}")
            return

        if isinstance(data, Image.Image):
            self._add_panel_from_pil(data.convert("RGBA"), source_path="clipboard")
            self.regenerate_labels()
            self.redraw()
            self.set_status("Pasted image from clipboard")
            self.auto_grid_if_reasonable()
            return

        if isinstance(data, list):
            self.load_paths([str(p) for p in data])
            return

        messagebox.showinfo("Clipboard", "No image or file list found in clipboard.")

    def _add_panel_from_pil(self, img: Image.Image, source_path: Optional[str] = None):
        ow, oh = img.size
        w, h = fit_size_keep_aspect(ow, oh, 350, 260)

        offset = 20 * (len(self.items) % 10)
        panel = PanelItem(
            id=self.get_next_id(),
            source_path=source_path,
            pil_image=img,
            original_size=(ow, oh),
            x=50 + offset,
            y=50 + offset,
            w=w,
            h=h,
            label="",
            show_label=True,
            label_font_size=self.default_label_size.get(),
            z_index=len(self.items),
        )
        self.items.append(panel)

    def add_text_box(self, default_text="Text"):
        item = TextItem(
            id=self.get_next_id(),
            text=default_text,
            x=60 + 15 * (len(self.items) % 10),
            y=60 + 15 * (len(self.items) % 10),
            w=320,
            h=90,
            font_size=self.default_text_size.get(),
            z_index=len(self.items),
        )
        self.items.append(item)
        self.selected_id = item.id
        self.refresh_selected_panel()
        self.redraw()
        self.set_status("Added text box")

    def redraw(self):
        self.canvas.delete("all")
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))

        if self.show_grid.get():
            self._draw_grid()

        self.canvas.create_rectangle(0, 0, self.canvas_w, self.canvas_h, fill=self.bg_color, outline="")

        for item in sorted(self.items, key=lambda x: x.z_index):
            if isinstance(item, PanelItem):
                self._draw_panel(item)
            elif isinstance(item, TextItem):
                self._draw_text_item(item)

        sel = self.get_selected()
        if sel is not None:
            self._draw_selection(sel)

    def _draw_grid(self):
        step = 50
        for x in range(0, self.canvas_w + 1, step):
            self.canvas.create_line(x, 0, x, self.canvas_h, fill="#eeeeee")
        for y in range(0, self.canvas_h + 1, step):
            self.canvas.create_line(0, y, self.canvas_w, y, fill="#eeeeee")

    def _draw_panel(self, item: PanelItem):
        if item.pil_image is None:
            return
        preview = item.pil_image.resize((max(1, item.w), max(1, item.h)), Image.LANCZOS)
        item.tk_preview = ImageTk.PhotoImage(preview)
        self.canvas.create_image(item.x, item.y, image=item.tk_preview, anchor="nw")

        if item.border_width > 0:
            self.canvas.create_rectangle(item.x, item.y, item.x + item.w, item.y + item.h,
                                         outline=item.border_color, width=item.border_width)

        if item.show_label and item.label:
            self.canvas.create_text(
                item.x + item.label_offset_x,
                item.y + item.label_offset_y,
                text=item.label,
                anchor="nw",
                font=("Arial", item.label_font_size, "bold"),
                fill="black"
            )

    def _draw_text_item(self, item: TextItem):
        if item.background:
            self.canvas.create_rectangle(item.x, item.y, item.x + item.w, item.y + item.h,
                                         fill=item.background, outline=item.outline or "")
        anchor_map = {"left": "nw", "center": "n", "right": "ne"}
        tx = item.x + 6 if item.align == "left" else item.x + item.w // 2 if item.align == "center" else item.x + item.w - 6
        self.canvas.create_text(
            tx, item.y + 6,
            text=item.text,
            anchor=anchor_map.get(item.align, "nw"),
            width=max(20, item.w - 12),
            font=("Arial", item.font_size, "bold" if item.bold else "normal"),
            fill=item.fill
        )
        if item.outline:
            self.canvas.create_rectangle(item.x, item.y, item.x + item.w, item.y + item.h, outline=item.outline, width=1)

    def _draw_selection(self, item: CanvasItem):
        x1, y1, x2, y2 = item.bbox()
        self.canvas.create_rectangle(x1, y1, x2, y2, outline="#2a73ff", width=2, dash=(5, 3))
        self.canvas.create_rectangle(x2 - HANDLE_SIZE, y2 - HANDLE_SIZE, x2, y2, fill="#2a73ff", outline="#2a73ff")

    def get_item_at(self, x: int, y: int) -> Optional[CanvasItem]:
        for item in sorted(self.items, key=lambda z: z.z_index, reverse=True):
            if item.contains(x, y):
                return item
        return None

    def get_selected(self) -> Optional[CanvasItem]:
        if self.selected_id is None:
            return None
        for item in self.items:
            if item.id == self.selected_id:
                return item
        return None

    def on_canvas_press(self, event):
        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))
        item = self.get_item_at(x, y)

        self.drag_start = (x, y)
        self.drag_mode = None
        self.orig_geom = None

        if item is None:
            self.selected_id = None
            self.refresh_selected_panel()
            self.redraw()
            return

        self.selected_id = item.id
        self.refresh_selected_panel()

        if item.resize_handle_hit(x, y):
            self.drag_mode = "resize"
        else:
            self.drag_mode = "move"

        self.orig_geom = (item.x, item.y, item.w, item.h)
        self.redraw()

    def on_canvas_drag(self, event):
        sel = self.get_selected()
        if sel is None or self.drag_mode is None or self.orig_geom is None:
            return

        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))
        dx = x - self.drag_start[0]
        dy = y - self.drag_start[1]

        ox, oy, ow, oh = self.orig_geom

        if self.drag_mode == "move":
            sel.x = max(0, ox + dx)
            sel.y = max(0, oy + dy)
        elif self.drag_mode == "resize":
            sel.w = max(MIN_PANEL_SIZE, ow + dx)
            sel.h = max(MIN_PANEL_SIZE, oh + dy)

        self.refresh_selected_panel(live=True)
        self.redraw()

    def on_canvas_release(self, event):
        self.drag_mode = None
        self.orig_geom = None

    def on_double_click(self, event):
        x = int(self.canvas.canvasx(event.x))
        y = int(self.canvas.canvasy(event.y))
        item = self.get_item_at(x, y)
        if isinstance(item, TextItem):
            new_text = simpledialog.askstring("Edit text", "Text:", initialvalue=item.text, parent=self.root)
            if new_text is not None:
                item.text = new_text
                self.refresh_selected_panel()
                self.redraw()
        elif isinstance(item, PanelItem):
            new_label = simpledialog.askstring("Edit label", "Panel label:", initialvalue=item.label, parent=self.root)
            if new_label is not None:
                item.label = new_label
                self.refresh_selected_panel()
                self.redraw()

    def refresh_selected_panel(self, live=False):
        sel = self.get_selected()
        if sel is None:
            self.sel_type_var.set("-")
            self.sel_label_var.set("")
            self.sel_text_var.set("")
            self.sel_x_var.set("0")
            self.sel_y_var.set("0")
            self.sel_w_var.set("0")
            self.sel_h_var.set("0")
            self.sel_font_var.set(str(DEFAULT_TEXT_FONT_SIZE))
            self.sel_show_label_var.set(True)
            return

        self.sel_type_var.set(sel.kind)
        self.sel_x_var.set(str(sel.x))
        self.sel_y_var.set(str(sel.y))
        self.sel_w_var.set(str(sel.w))
        self.sel_h_var.set(str(sel.h))

        if isinstance(sel, PanelItem):
            self.sel_label_var.set(sel.label)
            self.sel_text_var.set("")
            self.sel_font_var.set(str(sel.label_font_size))
            self.sel_show_label_var.set(sel.show_label)
        elif isinstance(sel, TextItem):
            self.sel_label_var.set("")
            self.sel_text_var.set(sel.text)
            self.sel_font_var.set(str(sel.font_size))
            self.sel_show_label_var.set(True)

        if not live:
            self.set_status(f"Selected {sel.kind} #{sel.id}")

    def apply_selected_properties(self):
        sel = self.get_selected()
        if sel is None:
            return
        try:
            sel.x = int(self.sel_x_var.get())
            sel.y = int(self.sel_y_var.get())
            sel.w = max(MIN_PANEL_SIZE, int(self.sel_w_var.get()))
            sel.h = max(MIN_PANEL_SIZE, int(self.sel_h_var.get()))
            font_size = max(6, int(self.sel_font_var.get()))
        except ValueError:
            messagebox.showerror("Invalid input", "Position, size, and font size must be integers.")
            return

        if isinstance(sel, PanelItem):
            sel.label = self.sel_label_var.get()
            sel.label_font_size = font_size
            sel.show_label = self.sel_show_label_var.get()
        elif isinstance(sel, TextItem):
            sel.text = self.sel_text_var.get()
            sel.font_size = font_size

        self.redraw()
        self.set_status("Properties applied")

    def bring_to_front(self):
        sel = self.get_selected()
        if sel is None:
            return
        max_z = max((i.z_index for i in self.items), default=0)
        sel.z_index = max_z + 1
        self.redraw()

    def send_to_back(self):
        sel = self.get_selected()
        if sel is None:
            return
        min_z = min((i.z_index for i in self.items), default=0)
        sel.z_index = min_z - 1
        self.redraw()

    def duplicate_selected(self):
        sel = self.get_selected()
        if sel is None:
            return
        if isinstance(sel, PanelItem):
            dup = PanelItem(
                id=self.get_next_id(),
                source_path=sel.source_path,
                pil_image=sel.pil_image.copy() if sel.pil_image else None,
                original_size=sel.original_size,
                x=sel.x + 20,
                y=sel.y + 20,
                w=sel.w,
                h=sel.h,
                label=sel.label,
                show_label=sel.show_label,
                label_font_size=sel.label_font_size,
                label_offset_x=sel.label_offset_x,
                label_offset_y=sel.label_offset_y,
                border_width=sel.border_width,
                border_color=sel.border_color,
                z_index=max((i.z_index for i in self.items), default=0) + 1,
            )
        else:
            dup = TextItem(
                id=self.get_next_id(),
                text=sel.text,
                x=sel.x + 20,
                y=sel.y + 20,
                w=sel.w,
                h=sel.h,
                font_size=sel.font_size,
                bold=sel.bold,
                fill=sel.fill,
                align=sel.align,
                background=sel.background,
                outline=sel.outline,
                z_index=max((i.z_index for i in self.items), default=0) + 1,
            )
        self.items.append(dup)
        self.selected_id = dup.id
        self.refresh_selected_panel()
        self.redraw()

    def delete_selected(self):
        sel = self.get_selected()
        if sel is None:
            return
        self.items = [i for i in self.items if i.id != sel.id]
        self.selected_id = None
        self.refresh_selected_panel()
        self.redraw()
        self.set_status("Deleted selected item")

    def regenerate_labels(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if self.keep_label_order.get():
            panels = sorted(panels, key=lambda p: p.id)
        else:
            panels = sorted(panels, key=lambda p: (p.y, p.x))
        for idx, panel in enumerate(panels):
            panel.label = next_panel_label(idx)
            panel.label_font_size = self.default_label_size.get()
        self.redraw()
        self.set_status("Regenerated panel labels")

    def toggle_labels(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if not panels:
            return
        turn_on = any(not p.show_label for p in panels)
        for p in panels:
            p.show_label = turn_on
        self.redraw()

    def apply_default_label_size(self):
        size = self.default_label_size.get()
        for item in self.items:
            if isinstance(item, PanelItem):
                item.label_font_size = size
        self.redraw()

    def auto_grid_if_reasonable(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if 2 <= len(panels) <= 12:
            self.auto_grid()

    def auto_grid(self):
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        if not panels:
            return

        n = len(panels)
        cols = math.ceil(math.sqrt(n))
        rows = math.ceil(n / cols)

        margin = self.outer_margin.get()
        gap_x = self.gap_x.get()
        gap_y = self.gap_y.get()

        avail_w = self.canvas_w - 2 * margin - (cols - 1) * gap_x
        avail_h = self.canvas_h - 2 * margin - (rows - 1) * gap_y

        cell_w = max(50, avail_w // cols)
        cell_h = max(50, avail_h // rows)

        panels_sorted = sorted(panels, key=lambda p: p.id if self.keep_label_order.get() else (p.y, p.x))

        for idx, p in enumerate(panels_sorted):
            r = idx // cols
            c = idx % cols
            x = margin + c * (cell_w + gap_x)
            y = margin + r * (cell_h + gap_y)
            ow, oh = p.original_size
            new_w, new_h = fit_size_keep_aspect(ow, oh, cell_w, cell_h)
            p.x = x + (cell_w - new_w) // 2
            p.y = y + (cell_h - new_h) // 2
            p.w = new_w
            p.h = new_h

        self.redraw()
        self.set_status(f"Auto-arranged {n} panel(s)")

    def get_selected_panels_or_all(self) -> List[PanelItem]:
        panels = [i for i in self.items if isinstance(i, PanelItem)]
        return panels

    def distribute_h(self):
        panels = sorted(self.get_selected_panels_or_all(), key=lambda p: p.x)
        if len(panels) < 3:
            return
        left = min(p.x for p in panels)
        right = max(p.x + p.w for p in panels)
        total_w = sum(p.w for p in panels)
        gap = (right - left - total_w) / (len(panels) - 1)
        x = left
        for p in panels:
            p.x = int(round(x))
            x += p.w + gap
        self.redraw()

    def distribute_v(self):
        panels = sorted(self.get_selected_panels_or_all(), key=lambda p: p.y)
        if len(panels) < 3:
            return
        top = min(p.y for p in panels)
        bottom = max(p.y + p.h for p in panels)
        total_h = sum(p.h for p in panels)
        gap = (bottom - top - total_h) / (len(panels) - 1)
        y = top
        for p in panels:
            p.y = int(round(y))
            y += p.h + gap
        self.redraw()

    def align_left(self):
        panels = self.get_selected_panels_or_all()
        if len(panels) < 2:
            return
        x = min(p.x for p in panels)
        for p in panels:
            p.x = x
        self.redraw()

    def align_top(self):
        panels = self.get_selected_panels_or_all()
        if len(panels) < 2:
            return
        y = min(p.y for p in panels)
        for p in panels:
            p.y = y
        self.redraw()

    def same_widths(self):
        panels = self.get_selected_panels_or_all()
        if len(panels) < 2:
            return
        w = panels[0].w
        for p in panels[1:]:
            ow, oh = p.original_size
            new_w = w
            new_h = int(round(new_w * oh / ow)) if ow else p.h
            p.w = max(MIN_PANEL_SIZE, new_w)
            p.h = max(MIN_PANEL_SIZE, new_h)
        self.redraw()

    def same_heights(self):
        panels = self.get_selected_panels_or_all()
        if len(panels) < 2:
            return
        h = panels[0].h
        for p in panels[1:]:
            ow, oh = p.original_size
            new_h = h
            new_w = int(round(new_h * ow / oh)) if oh else p.w
            p.w = max(MIN_PANEL_SIZE, new_w)
            p.h = max(MIN_PANEL_SIZE, new_h)
        self.redraw()

    def content_bbox(self) -> Optional[Tuple[int, int, int, int]]:
        if not self.items:
            return None
        x1 = min(i.x for i in self.items)
        y1 = min(i.y for i in self.items)
        x2 = max(i.x + i.w for i in self.items)
        y2 = max(i.y + i.h for i in self.items)
        return x1, y1, x2, y2

    def crop_canvas_to_content(self):
        bbox = self.content_bbox()
        if bbox is None:
            return
        margin = self.outer_margin.get()
        x1, y1, x2, y2 = bbox
        shift_x = margin - x1
        shift_y = margin - y1
        for item in self.items:
            item.x += shift_x
            item.y += shift_y
        self.canvas_w = max(100, x2 - x1 + 2 * margin)
        self.canvas_h = max(100, y2 - y1 + 2 * margin)
        self.canvas_w_entry.delete(0, "end")
        self.canvas_w_entry.insert(0, str(self.canvas_w))
        self.canvas_h_entry.delete(0, "end")
        self.canvas_h_entry.insert(0, str(self.canvas_h))
        self.canvas.config(scrollregion=(0, 0, self.canvas_w, self.canvas_h))
        self.redraw()
        self.set_status("Canvas cropped to content")

    def render_final_image(self) -> Image.Image:
        out = Image.new("RGB", (self.canvas_w, self.canvas_h), "white")
        draw = ImageDraw.Draw(out)

        for item in sorted(self.items, key=lambda x: x.z_index):
            if isinstance(item, PanelItem):
                if item.pil_image is None:
                    continue
                resized = item.pil_image.resize((item.w, item.h), Image.LANCZOS).convert("RGB")
                out.paste(resized, (item.x, item.y))
                if item.border_width > 0:
                    draw.rectangle([item.x, item.y, item.x + item.w, item.y + item.h],
                                   outline=item.border_color, width=item.border_width)
                if item.show_label and item.label:
                    font = get_default_font(item.label_font_size)
                    draw.text((item.x + item.label_offset_x, item.y + item.label_offset_y),
                              item.label, fill="black", font=font)

            elif isinstance(item, TextItem):
                if item.background:
                    draw.rectangle([item.x, item.y, item.x + item.w, item.y + item.h],
                                   fill=item.background, outline=item.outline or None)
                elif item.outline:
                    draw.rectangle([item.x, item.y, item.x + item.w, item.y + item.h],
                                   outline=item.outline, width=1)

                font = get_default_font(item.font_size)
                self._draw_wrapped_text_pil(draw, item, font)

        if self.auto_trim_on_export.get():
            out = self.trim_image(out, pad=self.outer_margin.get())
        return out

    def _draw_wrapped_text_pil(self, draw: ImageDraw.ImageDraw, item: TextItem, font):
        text = item.text
        max_width = max(20, item.w - 12)
        lines = self.wrap_text_to_width(draw, text, font, max_width)

        line_heights = []
        for line in lines:
            bbox = draw.textbbox((0, 0), line, font=font)
            line_heights.append(bbox[3] - bbox[1])

        y = item.y + 6
        for line, lh in zip(lines, line_heights):
            bbox = draw.textbbox((0, 0), line, font=font)
            tw = bbox[2] - bbox[0]
            if item.align == "center":
                x = item.x + (item.w - tw) / 2
            elif item.align == "right":
                x = item.x + item.w - 6 - tw
            else:
                x = item.x + 6
            draw.text((x, y), line, fill=item.fill, font=font)
            y += lh + 4

    @staticmethod
    def wrap_text_to_width(draw: ImageDraw.ImageDraw, text: str, font, max_width: int) -> List[str]:
        words = text.split()
        if not words:
            return [""]
        lines = []
        current = words[0]
        for word in words[1:]:
            test = current + " " + word
            bbox = draw.textbbox((0, 0), test, font=font)
            if bbox[2] - bbox[0] <= max_width:
                current = test
            else:
                lines.append(current)
                current = word
        lines.append(current)
        return lines

    @staticmethod
    def trim_image(img: Image.Image, pad: int = 0) -> Image.Image:
        bg = Image.new(img.mode, img.size, img.getpixel((0, 0)))
        diff = ImageOps.invert(ImageChopsSafe.difference(img, bg).convert("L"))
        bbox = ImageOps.invert(diff).getbbox()
        if bbox is None:
            return img
        x1, y1, x2, y2 = bbox
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(img.width, x2 + pad)
        y2 = min(img.height, y2 + pad)
        return img.crop((x1, y1, x2, y2))

    def export_image(self, fmt: str):
        if not self.items:
            messagebox.showinfo("Nothing to export", "Add some panels or text first.")
            return

        ext_map = {"PNG": ".png", "TIFF": ".tiff"}
        path = filedialog.asksaveasfilename(
            title=f"Export {fmt}",
            defaultextension=ext_map[fmt],
            filetypes=[(f"{fmt} file", f"*{ext_map[fmt]}")]
        )
        if not path:
            return

        try:
            img = self.render_final_image()
            save_kwargs = {}
            if fmt == "TIFF":
                save_kwargs["compression"] = "tiff_deflate"
            img.save(path, format=fmt, dpi=(300, 300), **save_kwargs)
            self.set_status(f"Exported {fmt}: {path}")
        except Exception as e:
            messagebox.showerror("Export error", f"Could not export file.\n\n{e}")

    def export_pdf(self):
        if not self.items:
            messagebox.showinfo("Nothing to export", "Add some panels or text first.")
            return

        path = filedialog.asksaveasfilename(
            title="Export PDF",
            defaultextension=".pdf",
            filetypes=[("PDF file", "*.pdf")]
        )
        if not path:
            return

        try:
            img = self.render_final_image()
            rgb = img.convert("RGB")
            rgb.save(path, "PDF", resolution=300.0)
            self.set_status(f"Exported PDF: {path}")
        except Exception as e:
            messagebox.showerror("Export error", f"Could not export PDF.\n\n{e}")


class ImageChopsSafe:
    @staticmethod
    def difference(img1: Image.Image, img2: Image.Image) -> Image.Image:
        from PIL import ImageChops
        return ImageChops.difference(img1, img2)


def main():
    if DND_AVAILABLE:
        root = TkinterDnD.Tk()
    else:
        root = tk.Tk()

    try:
        style = ttk.Style(root)
        if "vista" in style.theme_names():
            style.theme_use("vista")
    except Exception:
        pass

    app = FigureBoardApp(root)
    if not DND_AVAILABLE:
        messagebox.showwarning(
            "Drag-and-drop unavailable",
            "tkinterdnd2 is not available.\n\nYou can still use the app with the 'Add image/PDF files' button.\n\nInstall with:\npip install tkinterdnd2"
        )
    root.mainloop()


if __name__ == "__main__":
    main()